# Native CBQM 小规模求解验证

这本 notebook 验证 `cbqm.v1 -> native solver -> cbqm-result.v1` 的原生路径，对比 `ExactCbqmSolver` 与 `LocalSearchCbqmSolver`，并通过独立 oracle 与 `ProblemCase` exact promotion 做交叉检查。算法、objective、feasibility 和搜索逻辑都在 Python 脚本中；这里仅导入、调用、展示与断言。

## 1. 定位仓库并导入

In [ ]:
import copy
import json
import sys
from pathlib import Path


PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "lib").is_dir() or not (PROJECT_ROOT / "problem").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate the QSolutionData repository root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lib.contracts import validate_cbqm, validate_cbqm_result
from lib.solvers.cbqm import ExactCbqmSolver, LocalSearchCbqmSolver
from problem.benchmarks import build_cbqm_mathematical_suite
from problem import (
    ProblemArtifact,
    ProblemCase,
    TaskDefinition,
    solve_native_problem_task,
)
from tests.oracles.cbqm import enumerate_cbqm_feasible
from tests.oracles.numbers import public_json_number

## 2. 读取 canonical CBQM

直接复用契约示例，避免 notebook 再维护一份模型定义。

In [ ]:
example_path = PROJECT_ROOT / "contracts" / "examples" / "cbqm.v1.example.json"
problem = json.loads(example_path.read_text(encoding="utf-8"))
original_problem = copy.deepcopy(problem)

validate_cbqm(problem)
print("Problem:", problem["problem_id"])
print("Variables:", [variable["name"] for variable in problem["variables"]])
print("Constraints:", [constraint["name"] for constraint in problem["constraints"]])

## 3. 独立 oracle 与原生 Exact solver

`tests.oracles` 不调用 production evaluator 或 solver。两条路径应得到相同的 sample 与原始 CBQM objective。

In [ ]:
oracle_rows = enumerate_cbqm_feasible(problem)
oracle_sample = oracle_rows[0]["sample"]
oracle_objective = public_json_number(oracle_rows[0]["objective_exact"])

result = ExactCbqmSolver().solve(problem)
validate_cbqm_result(problem, result)

assert result["status"] == "optimal"
assert result["best_sample"] == oracle_sample
assert result["best_objective"] == oracle_objective
assert result["feasibility"]["feasible"] is True
assert problem == original_problem

print("Oracle optimum:", oracle_sample, oracle_objective)
print("Solver bounds:", result["bounds"])
print("Solver proof claim:", result["proof"])

## 4. 不可行模型

把两个变量都固定为 0，与 `select_one` 约束冲突。完整枚举后应返回 `infeasible`，且候选三元组全部为 `null`。

In [ ]:
infeasible_problem = copy.deepcopy(problem)
infeasible_problem["problem_id"] = "two-asset-infeasible"
infeasible_problem["fixed_values"] = [
    {"index": 0, "value": 0},
    {"index": 1, "value": 0},
]

infeasible_result = ExactCbqmSolver().solve(infeasible_problem)
validate_cbqm_result(infeasible_problem, infeasible_result)

assert infeasible_result["status"] == "infeasible"
assert infeasible_result["best_sample"] is None
assert infeasible_result["best_objective"] is None
assert infeasible_result["feasibility"] is None
assert infeasible_result["proof"]["claim"] == "infeasibility"

print("Infeasibility proof:", infeasible_result["proof"])

## 5. ProblemCase 原生执行与 exact promotion

`cbqm-result.v1` 中的 `optimal` 只是 solver 声明。写入 task 的 `exact=True` 前，`solve_native_problem_task()` 会在配置上限内独立枚举 canonical CBQM。

In [ ]:
case = ProblemCase(
    problem_id=problem["problem_id"],
    artifacts=(
        ProblemArtifact(
            artifact_id="cbqm",
            representation="cbqm.v1",
            payload=problem,
        ),
    ),
    tasks=(
        TaskDefinition(
            task_id="select-one",
            canonical_artifact_id="cbqm",
            sense="minimize",
        ),
    ),
)

record = solve_native_problem_task(
    case,
    task_id="select-one",
    artifact_id="cbqm",
    solver=ExactCbqmSolver(),
    update_best=True,
    exact_verification_max_variables=24,
)

assert record.canonical_solution == oracle_sample
assert record.canonical_objective_value == oracle_objective
assert record.exact_for_task is True
assert record.update.current.exact is True
assert record.update.current.metadata["exactness"]["independent_cbqm_optimality_verified"] is True

print("Persistent exact:", record.exact_for_task)
print("Exactness evidence:", record.update.current.metadata["exactness"])

## 6. 原生 Exact 与 Local Search 对比

下面直接消费数学 benchmark 生成的 `cbqm.v1`。Local Search 必须给出通过契约校验的可行解，但只声明 `feasible`；当前固定 seed 在这些微型案例上命中 exact objective，并不构成一般性的最优性证明。

In [ ]:
comparison_rows = []
for benchmark_name, benchmark_problem in build_cbqm_mathematical_suite().items():
    exact_result = ExactCbqmSolver().solve(benchmark_problem)
    local_result = LocalSearchCbqmSolver().solve(
        benchmark_problem,
        {"seed": 0, "max_restarts": 4},
    )
    validate_cbqm_result(benchmark_problem, local_result)

    assert local_result["status"] == "feasible"
    assert local_result["feasibility"]["feasible"] is True
    assert "proof" not in local_result
    assert local_result["best_objective"] == exact_result["best_objective"]

    comparison_rows.append({
        "benchmark": benchmark_name,
        "exact": exact_result["best_objective"],
        "local_search": local_result["best_objective"],
        "local_status": local_result["status"],
        "starts": local_result["metrics"]["starts_attempted"],
    })

for row in comparison_rows:
    print(row)

## 结论与边界

- 原生 CBQM solver 返回原始 objective、feasibility、bounds 与 proof，不借用 QUBO penalty energy。
- Exact solver 的主流程只枚举 fixed values 之外的自由变量；`max_variables` 仍是显式安全上限。
- Local Search 先修复原始约束、再在可行域内优化；没有可行候选时只返回 `unknown`，不会误报 `infeasible`。
- result validator 会重算候选与 trace，但不会把任意 backend 的 `optimal` 自动视为永久证明。
- ProblemCase exact promotion 通过第二次独立 CBQM 穷举完成；超过验证上限时仍保留候选，但保持非 exact。
- timeout、伪造声明与输入不变性的更细边界由自动化测试覆盖，notebook 不复制这些测试 helper。